In [3]:
from pathlib import Path
import pandas as pd
import cv2
import numpy as np
import os
from skimage.measure import shannon_entropy
from skimage.filters import threshold_otsu
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor

def made_csv(root_dir: str | Path, output_csv_path: str | Path):
    root_dir = Path(root_dir)
    
    data_list = []
    
    for file_path in root_dir.rglob("*.png"):
        # NYBD1-A2_250nM_BT549_site1_FITC_Day0.png
        # or NYBD1-A2_250nM_BT549_site1_Hoechst.png
        info_list = file_path.stem.lower().split("_")
        if len(info_list) == 5:
            compound, concentration, cell_line, site, channel = info_list
        else:
            compound, concentration, cell_line, site, channel, day = info_list
        compound = compound.replace("note-",'')
        # NCI60 Images/breast/bt549/nybd1/filename.png
        organ = file_path.parts[-4]
        rel_path = file_path.relative_to(root_dir)

        data_list.append({
            "file_name": file_path.name,
            "organ": organ,
            "cell_line": cell_line,
            "compound": compound,
            "concentration": concentration,
            "channel": channel,
            "rel_path": str(rel_path)

        })

    df = pd.DataFrame(data_list)
    df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
    print(f"saving complete to {output_csv_path}, total {len(df)} numbers file")



# OpenCV 자체의 멀티스레딩이 파이썬 멀티프로세싱과 충돌해 데드락(Deadlock)이 걸리는 것을 방지
cv2.setNumThreads(0) 

def calculate_metrics(image_path: Path):
    """단일 이미지의 Mean, SBR, Entropy를 계산하는 함수 (변경 없음)"""
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE | cv2.IMREAD_ANYDEPTH)
    
    if img is None:
        return np.nan, np.nan, np.nan
        
    mean_val = np.mean(img)
    entropy_val = shannon_entropy(img)
    
    if img.min() == img.max():
        sbr_val = 1.0
    else:
        try:
            th = threshold_otsu(img)
            signal = img[img > th]
            background = img[img <= th]
            
            mean_signal = np.mean(signal) if len(signal) > 0 else 0
            mean_bg = np.mean(background) if len(background) > 0 else 0
            
            if mean_bg == 0:
                sbr_val = float('inf') if mean_signal > 0 else 0.0
            else:
                sbr_val = mean_signal / mean_bg
        except Exception:
            sbr_val = np.nan
            
    return mean_val, sbr_val, entropy_val

def update_csv_parallel(input_csv: str, root_dir: str, output_csv: str):
    df = pd.read_csv(input_csv)
    root = Path(root_dir)
    
    full_paths = [root / rel_path for rel_path in df["rel_path"]]
    
    # 2. 서버의 가용 CPU 코어 수 확인
    max_cores = os.cpu_count()
    print(f"🚀 서버의 CPU 코어 {max_cores}개를 모두 사용하여 병렬 연산을 시작합니다...")
    
    # 3. 멀티프로세싱 풀(Pool) 생성 및 실행
    with ProcessPoolExecutor(max_workers=max_cores) as executor:
        # executor.map은 입력된 리스트의 순서를 그대로 보장하며 결과를 반환합니다.
        # tqdm과 결합하여 병렬 처리 진행 상황을 실시간으로 확인
        results = list(tqdm(executor.map(calculate_metrics, full_paths, chunksize=1000), total=len(full_paths)))
    
    # 4. 순서가 보장된 결과값을 분리하여 리스트로 정리
    means = [res[0] for res in results]
    sbrs = [res[1] for res in results]
    entropies = [res[2] for res in results]
    
    # 데이터프레임에 칼럼 추가
    df["mean"] = means
    df["sbr"] = sbrs
    df["entropy"] = entropies
    
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"\n✅ 고속 연산 완료! [{output_csv}]에 저장되었습니다.")

# ==========================================
# 실행 부분
# ==========================================
if __name__ == "__main__":
    made_csv(root_dir="../data/png_converted", output_csv_path="../data/png_converted/NCI60_Images.csv")
    INPUT_CSV = "../data/png_converted/NCI60_Images.csv"
    DATA_ROOT = "../data/png_converted"
    OUTPUT_CSV = "../data/png_converted/NCI60_Images.csv"
    
    update_csv_parallel(INPUT_CSV, DATA_ROOT, OUTPUT_CSV)

saving complete to ../data/png_converted/NCI60_Images.csv, total 34696 numbers file
🚀 서버의 CPU 코어 64개를 모두 사용하여 병렬 연산을 시작합니다...


100%|██████████| 34696/34696 [02:23<00:00, 242.23it/s] 



✅ 고속 연산 완료! [../data/png_converted/NCI60_Images.csv]에 저장되었습니다.
